# CIS 635 — NPS visitation × NOAA GHCN-Daily weather
## Notebook 01: data acquisition, cleaning, integration, and first look

This notebook runs the `npsweather` package from the project repository. It does the following:
1. Downloads the NPS visitation data and NOAA weather-station data.
2. Matches each park to a nearby weather station.
3. Builds one park-month table that joins visits with that month's weather.

**To rerun:** choose *Runtime → Run all*. No API key is needed, because park coordinates come from a snapshot committed in the repo. The first run downloads about 330 station files, which takes roughly 5 minutes.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/kowshid/cis-635-project"
USE_DRIVE = True  # keep data on Google Drive between sessions; False = temporary Colab disk

if IN_COLAB:
    PROJECT = Path("/content/cis635-nps-weather")
    if PROJECT.exists():
        subprocess.run(["git", "-C", str(PROJECT), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(PROJECT)], check=True)

    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["NPSW_DATA_DIR"] = "/content/drive/MyDrive/cis635/data"
    else:
        os.environ["NPSW_DATA_DIR"] = "/content/data"

    try:  # optional: only needed to refresh park coordinates from the live API
        from google.colab import userdata
        os.environ["NPS_API_KEY"] = userdata.get("NPS_API_KEY")
    except Exception:
        print("No NPS_API_KEY secret; using the committed park-coordinate snapshot.")
else:
    PROJECT = Path.cwd().resolve().parent  # notebooks/ -> project root

sys.path.insert(0, str(PROJECT / "src"))

In [ ]:
import json
import logging

import matplotlib.pyplot as plt
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

from npsweather import config, pipeline

config.ensure_dirs()
print("Data directory:", config.DATA_DIR)
print("Study period:", config.STUDY_START_YEAR, "-", config.STUDY_END_YEAR)

### Step 1: Inspect the NPS visitation file
This step downloads `Main_Data.csv` (NPS Visitor Use Statistics, 1979–2025) and shows how its columns were interpreted.

In [ ]:
pipeline.inspect_visits()

### Step 2: Run the pipeline
The stages run in order: `visits` → `coords` → `stations` → `match` → `join`. Each stage caches its output, so reruns are fast.

In [ ]:
pipeline.run(["all"])

### Step 3: Data quality

In [ ]:
pm = pd.read_parquet(config.PARK_MONTH_FILE)
match = pd.read_parquet(config.MATCH_FILE)
report = json.loads(config.QUALITY_REPORT_FILE.read_text())

print(f"park-month table: {pm.shape[0]:,} rows x {pm.shape[1]} columns")
pd.Series({k: v for k, v in report.items() if not isinstance(v, dict)}, name="value").to_frame()

In [ ]:
summary = pd.concat(
    {
        "station match quality": pd.Series(report["match_quality"]),
        "park-month status": pd.Series(report["status_counts"]),
    }
).rename("count").to_frame()
summary

In [ ]:
# Park-month status by year: zero-visit months that don't fit a park's usual
# seasonal pattern ("isolated_zero"), e.g. closures, show up here.
status_by_year = pm.pivot_table(index="year", columns="status", values="unit_code", aggfunc="count", fill_value=0)
status_by_year.loc[2015:]

### Step 4: First look

In [ ]:
coords = pd.read_parquet(config.COORDS_FILE)
mq = match.merge(coords[["unit_code", "latitude", "longitude"]], on="unit_code", how="left")
lower48 = mq["longitude"].between(-125, -66) & mq["latitude"].between(24, 50)
counts = mq["match_quality"].value_counts()

fig, (ax_map, ax_hist) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.6, 1]})
for quality, grp in mq[lower48].groupby("match_quality"):
    ax_map.scatter(grp["longitude"], grp["latitude"], s=14, label=f"{quality} ({counts[quality]})")
ax_map.set(title="Park units by station-match quality (lower 48 shown)", xlabel="Longitude", ylabel="Latitude")
ax_map.legend(title="match quality (all units)", fontsize=8)

ax_hist.hist(mq["distance_km"].dropna(), bins=40, alpha=0.8)
ax_hist.axvline(config.GOOD_MATCH_KM, color="k", ls="--", lw=1, label=f"{config.GOOD_MATCH_KM:.0f} km")
ax_hist.set(title="Park-to-station distance", xlabel="km", ylabel="Parks")
ax_hist.legend()
plt.tight_layout()
plt.show()

In [ ]:
open_months = pm[pm["status"] == "positive"]
annual = open_months.groupby("year").agg(
    visits_millions=("rec_visits", lambda s: s.sum() / 1e6),
    reporting_units=("unit_code", "nunique"),
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(annual.index, annual["visits_millions"], marker="o")
ax.axvspan(2019.5, 2021.5, color="grey", alpha=0.2, label="COVID period")
ax.set(title="Total recreation visits, all study units", xlabel="Year", ylabel="Millions of visits")
ax.legend()
plt.tight_layout()
plt.show()
annual.tail()

In [ ]:
EXAMPLE_PARK = "YELL"  # any unit code, e.g. GRCA, ACAD, ZION, GRSM

codes = set(pm["unit_code"])
code = EXAMPLE_PARK if EXAMPLE_PARK in codes else pm.groupby("unit_code")["rec_visits"].mean().idxmax()
ex = pm[pm["unit_code"] == code].sort_values("date")
years = sorted(y for y in ex["year"].unique() if y < 2020)[-5:]
ex = ex[ex["year"].isin(years)]
name = ex["park_name"].dropna().iloc[0] if "park_name" in ex and ex["park_name"].notna().any() else code

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(ex["date"], ex["rec_visits"] / 1e3, width=25, color="tab:green", alpha=0.6, label="Recreation visits")
ax1.set_ylabel("Visits (thousands)")
ax2 = ax1.twinx()
ax2.plot(ex["date"], ex["tavg_mean_c"], color="tab:red", marker="o", ms=3, label="Mean temperature")
ax2.set_ylabel("Mean temperature (°C)")
ax1.set_title(f"{name} ({code}): monthly visits vs. matched-station temperature, {years[0]}–{years[-1]}")
fig.legend(loc="upper left", bbox_to_anchor=(0.07, 0.92))
plt.tight_layout()
plt.show()

### Next
Exploratory analysis comes next, in notebook 02. It covers distributions, trends, seasonality profiles, and weather–visitation correlations. It will also settle which units to exclude, for example far matches and linear units such as trails and parkways.